# 統計製程管制（SPC）概論與變異分析

> **課程定位**：理論架構篇（1/6）  
> **學習路徑**：01 概論 → 02 X-bar & R → 03 I-MR → 04 P Chart → 05 製程能力 → 06 綜合案例

## 學習目標
- 理解 SPC 的定義、歷史與在智慧製造中的角色
- 區分普通原因變異與特殊原因變異
- 掌握常態分配與 3-Sigma 原則
- 認識控制圖的基本結構

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# 設定中文字型
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'Microsoft JhengHei', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100

## 1. 什麼是統計製程管制（SPC）？

**統計製程管制（Statistical Process Control, SPC）** 是一種利用統計方法監控和控制製程的品質管理工具，由 **Walter A. Shewhart** 於 1920 年代在貝爾實驗室首創。

### 歷史脈絡

| 年代 | 里程碑 | 代表人物 |
|------|--------|----------|
| 1924 | 發明第一張控制圖 | Walter Shewhart |
| 1950s | 將 SPC 引入日本製造業 | W. Edwards Deming |
| 1980s | 六標準差（Six Sigma）運動 | Motorola / GE |
| 2010s+ | Industry 4.0：即時感測器 + 自動化 SPC | 智慧製造整合 |

### SPC 在智慧製造中的角色

在 Industry 4.0 環境下，SPC 不再僅是事後抽檢，而是：
- **即時監控**：透過 IoT 感測器持續收集數據
- **自動警報**：當製程偏移時自動觸發通知
- **預測維護**：結合趨勢分析預測設備異常
- **數據驅動決策**：以統計證據取代經驗判斷

## 2. 變異的本質

製程中的變異分為兩大類：

### 普通原因變異（Common Cause Variation）
- 製程固有的**隨機波動**
- 來源：材料微小差異、環境溫濕度波動、量測誤差
- 特性：**可預測**、呈常態分配
- 對策：需要**改變系統**才能減少

### 特殊原因變異（Special Cause Variation）
- **非隨機**的異常偏移
- 來源：設備故障、操作錯誤、原料批次異常
- 特性：**不可預測**、會破壞分配型態
- 對策：需要**找出並消除**特定原因

> **知識補給站**  
> Deming 估計，製程問題中約 **94% 來自普通原因**（系統問題），只有 **6% 來自特殊原因**。這意味著大多數品質改善需要管理層改變系統，而非責怪操作員。

In [ ]:
np.random.seed(42)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 普通原因變異：穩定製程
stable_process = np.random.normal(loc=500, scale=2, size=100)
axes[0].plot(stable_process, 'b-o', markersize=3, alpha=0.7)
axes[0].axhline(y=500, color='green', linestyle='--', label='中心線 (\u03bc=500)')
axes[0].axhline(y=506, color='red', linestyle='--', label='UCL (\u03bc+3\u03c3)')
axes[0].axhline(y=494, color='red', linestyle='--', label='LCL (\u03bc-3\u03c3)')
axes[0].fill_between(range(100), 494, 506, alpha=0.1, color='green')
axes[0].set_title('普通原因變異：穩定製程', fontsize=14, fontweight='bold')
axes[0].set_xlabel('樣本編號')
axes[0].set_ylabel('量測值 (\u03bcm)')
axes[0].legend(loc='upper right', fontsize=9)
axes[0].set_ylim(490, 512)

# 特殊原因變異：製程偏移
shifted_process = np.concatenate([
    np.random.normal(loc=500, scale=2, size=60),
    np.random.normal(loc=504, scale=2, size=40)
])
axes[1].plot(shifted_process, 'b-o', markersize=3, alpha=0.7)
axes[1].axhline(y=500, color='green', linestyle='--', label='中心線 (\u03bc=500)')
axes[1].axhline(y=506, color='red', linestyle='--', label='UCL')
axes[1].axhline(y=494, color='red', linestyle='--', label='LCL')
axes[1].fill_between(range(100), 494, 506, alpha=0.1, color='green')
axes[1].axvline(x=60, color='orange', linestyle=':', linewidth=2, label='偏移發生點')
axes[1].set_title('特殊原因變異：製程偏移', fontsize=14, fontweight='bold')
axes[1].set_xlabel('樣本編號')
axes[1].set_ylabel('量測值 (\u03bcm)')
axes[1].legend(loc='upper right', fontsize=9)
axes[1].set_ylim(490, 512)

plt.tight_layout()
plt.show()

print("穩定製程 - 超出管制界限的點數:", np.sum((stable_process > 506) | (stable_process < 494)))
print("偏移製程 - 超出管制界限的點數:", np.sum((shifted_process > 506) | (shifted_process < 494)))

## 3. 常態分配與 3-Sigma 原則

SPC 的核心假設：當製程僅受普通原因影響時，數據近似**常態分配（Normal Distribution）**。

### 3-Sigma 原則

$$X \sim N(\mu, \sigma^2)$$

| 範圍 | 涵蓋機率 | 製程意義 |
|------|----------|----------|
| $\mu \pm 1\sigma$ | 68.27% | 約 2/3 的數據 |
| $\mu \pm 2\sigma$ | 95.45% | 幾乎所有數據 |
| $\mu \pm 3\sigma$ | 99.73% | 僅 0.27% 超出 |

> **口訣**：「68-95-99.7 法則」—— 記住這三個數字就掌握了常態分配的精髓。

In [ ]:
x = np.linspace(-4, 4, 1000)
y = stats.norm.pdf(x)

fig, ax = plt.subplots(figsize=(12, 6))

# 繪製 PDF 曲線
ax.plot(x, y, 'k-', linewidth=2)

# 填充不同 sigma 區域
colors = ['#2196F3', '#4CAF50', '#FF9800']
labels = ['\u00b11\u03c3 (68.27%)', '\u00b12\u03c3 (95.45%)', '\u00b13\u03c3 (99.73%)']
sigmas = [1, 2, 3]

for sigma, color, label in zip(reversed(sigmas), reversed(colors), reversed(labels)):
    ax.fill_between(x, y, where=(x >= -sigma) & (x <= sigma), 
                     alpha=0.3, color=color, label=label)

# 標記 sigma 位置
for s in range(-3, 4):
    if s != 0:
        ax.axvline(x=s, color='gray', linestyle=':', alpha=0.5)
        ax.text(s, -0.02, f'{s}\u03c3', ha='center', fontsize=10)

ax.set_title('常態分配與 3-Sigma 原則', fontsize=16, fontweight='bold')
ax.set_xlabel('標準差 (\u03c3)', fontsize=12)
ax.set_ylabel('機率密度', fontsize=12)
ax.legend(fontsize=12, loc='upper right')
ax.set_ylim(-0.05, 0.45)
plt.tight_layout()
plt.show()

# 計算各區間的精確機率
for sigma in [1, 2, 3]:
    prob = stats.norm.cdf(sigma) - stats.norm.cdf(-sigma)
    outside = 1 - prob
    ppm = outside * 1_000_000
    print(f"\u00b1{sigma}\u03c3: 涵蓋 {prob*100:.2f}%, 超出 {outside*100:.4f}%, 相當於 {ppm:.0f} PPM")

## 4. 控制圖的基本結構

控制圖是 SPC 的核心工具，由以下三條線組成：

- **中心線（CL, Center Line）**：製程平均值 $\bar{X}$
- **管制上限（UCL, Upper Control Limit）**：$CL + 3\sigma$
- **管制下限（LCL, Lower Control Limit）**：$CL - 3\sigma$

### 控制圖類型總覽

| 類別 | 圖表名稱 | 數據類型 | 適用情境 |
|------|----------|----------|----------|
| **計量型** | X-bar & R 圖 | 連續型，有子組 | 每批取多個樣本（n=2~10） |
| **計量型** | I-MR 圖 | 連續型，個別值 | 每次僅一個量測值 |
| **計數型** | P 圖 | 比例（不良率） | 合格/不合格判定 |
| **計數型** | C 圖 | 計數（缺陷數） | 固定面積/長度的缺陷數 |

> **知識補給站**  
> 控制圖的管制界限（UCL/LCL）是由**數據計算**得出，與規格界限（USL/LSL）不同。規格界限由**客戶或工程需求**決定。兩者混淆是初學者最常見的錯誤。

In [ ]:
np.random.seed(123)
data = np.random.normal(loc=50, scale=2, size=30)
# 注入一個異常點
data[22] = 58

mean = np.mean(data)
std = np.std(data, ddof=1)
ucl = mean + 3 * std
lcl = mean - 3 * std

fig, ax = plt.subplots(figsize=(14, 6))

# 繪製數據點
for i, val in enumerate(data):
    color = 'red' if val > ucl or val < lcl else 'blue'
    marker = 's' if val > ucl or val < lcl else 'o'
    ax.plot(i+1, val, marker=marker, color=color, markersize=6, zorder=5)

ax.plot(range(1, 31), data, 'b-', alpha=0.5, linewidth=1)

# 管制線
ax.axhline(y=mean, color='green', linestyle='-', linewidth=2, label=f'CL = {mean:.2f}')
ax.axhline(y=ucl, color='red', linestyle='--', linewidth=2, label=f'UCL = {ucl:.2f}')
ax.axhline(y=lcl, color='red', linestyle='--', linewidth=2, label=f'LCL = {lcl:.2f}')

# sigma 區域
for k, alpha_val in [(1, 0.15), (2, 0.10), (3, 0.05)]:
    ax.fill_between(range(0, 32), mean - k*std, mean + k*std, alpha=alpha_val, color='blue')

# 標註
ax.annotate('異常點！', xy=(23, data[22]), xytext=(25, data[22]+1),
            arrowprops=dict(arrowstyle='->', color='red'), fontsize=12, color='red', fontweight='bold')

ax.set_title('控制圖基本結構示意', fontsize=16, fontweight='bold')
ax.set_xlabel('樣本編號', fontsize=12)
ax.set_ylabel('量測值', fontsize=12)
ax.legend(fontsize=11, loc='upper left')
ax.set_xlim(0, 31)
plt.tight_layout()
plt.show()

## 5. 實務情境：半導體晶圓厚度監控

### 情境描述

某半導體晶圓廠監控晶圓研磨後的厚度，目標為 **500 \u03bcm**。

- 每小時抽取 **5 片晶圓** 量測厚度
- 共收集 **25 個子組**（subgroups）的數據
- 製程標準差約 **2 \u03bcm**

以下我們生成模擬數據，作為後續 Notebook 02（X-bar & R 圖）的基礎。

In [ ]:
np.random.seed(42)

n_subgroups = 25
subgroup_size = 5
target = 500
sigma = 2

# 生成穩定製程數據
wafer_data = np.random.normal(loc=target, scale=sigma, size=(n_subgroups, subgroup_size))

# 建立 DataFrame
columns = [f'晶圓_{i+1}' for i in range(subgroup_size)]
df_wafer = pd.DataFrame(wafer_data, columns=columns)
df_wafer.index = [f'子組_{i+1:02d}' for i in range(n_subgroups)]
df_wafer.index.name = '子組編號'

# 計算基本統計量
df_wafer['子組平均'] = df_wafer[columns].mean(axis=1)
df_wafer['子組全距'] = df_wafer[columns].max(axis=1) - df_wafer[columns].min(axis=1)

print("=" * 60)
print("半導體晶圓厚度數據（前 10 組）")
print("=" * 60)
display(df_wafer.head(10).round(2))

print(f"\n整體統計摘要：")
print(f"  總平均 (X\u0304\u0304) = {df_wafer['子組平均'].mean():.2f} \u03bcm")
print(f"  平均全距 (R\u0304) = {df_wafer['子組全距'].mean():.2f} \u03bcm")
print(f"  數據筆數 = {n_subgroups} 組 \u00d7 {subgroup_size} 片 = {n_subgroups * subgroup_size} 筆")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# 子組平均趨勢
axes[0].plot(range(1, 26), df_wafer['子組平均'], 'b-o', markersize=5)
axes[0].axhline(y=target, color='green', linestyle='--', label=f'目標值 = {target} \u03bcm')
axes[0].set_title('子組平均值趨勢圖', fontsize=14, fontweight='bold')
axes[0].set_xlabel('子組編號')
axes[0].set_ylabel('平均厚度 (\u03bcm)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 子組全距趨勢
axes[1].plot(range(1, 26), df_wafer['子組全距'], 'r-s', markersize=5)
axes[1].set_title('子組全距趨勢圖', fontsize=14, fontweight='bold')
axes[1].set_xlabel('子組編號')
axes[1].set_ylabel('全距 (\u03bcm)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n從視覺上觀察，製程看起來穩定——我們將在 Notebook 02 用 X-bar & R 圖嚴謹驗證。")

## 6. 本章小結

| 概念 | 重點 |
|------|------|
| SPC 定義 | 利用統計方法監控製程，區分變異來源 |
| 普通原因 | 製程固有隨機波動，需改變系統 |
| 特殊原因 | 可識別的異常偏移，需找出並消除 |
| 3-Sigma 原則 | 99.73% 數據落在 \u00b13\u03c3 內 |
| 控制圖結構 | CL（中心線）、UCL / LCL（管制上下限） |

---

## 課堂練習

### 基礎題
1. 使用 `scipy.stats.norm` 計算數據落在 \u00b12\u03c3 範圍內的百分比，並與理論值比較。

### 進階題
2. 給定以下數據，判斷是否存在特殊原因變異：
   `[50.1, 49.8, 50.3, 50.0, 49.9, 50.2, 50.1, 53.5, 50.0, 49.7]`

### 挑戰題
3. 模擬一個具有**漸進趨勢**（tool wear）的製程：前 50 點正常，後 50 點每個點平均增加 0.02，繪製時序圖並觀察。

---

> **下一堂課**：[02_控制圖基礎_Xbar_R圖.ipynb](./02_控制圖基礎_Xbar_R圖.ipynb) — 學習繪製最基本的 X-bar & R 控制圖